# Detección de reproducciones impresas de billetes colombianosUniversidad Militar Nueva Granada — Ingeniería MultimediaJuan Castaño (1202720) · Nicolás Moreno (1202437) · Andrés Navia (1202517)---Este cuaderno reproduce la cadena completa de procesamiento.**No se emplea ninguna librería de visión artificial.** Todas las operacionesestán implementadas sobre arreglos numéricos a partir de sus definicionesmatemáticas: la extracción de la región del objeto mediante etiquetado decomponentes conexas, y la transformación de perspectiva mediante laestimación explícita de la homografía y el remapeo inverso.Ejecute las celdas en orden.

## 1. Obtener el código y las dependencias

In [ ]:
# Opción A: clonar desde GitHub#!git clone https://github.com/USUARIO/deteccion-billetes-colombianos.git#%cd deteccion-billetes-colombianos# Opción B: subir el comprimido del proyectofrom google.colab import filessubidos = files.upload()          # seleccione ProyectoBilletes.zip!unzip -q -o ProyectoBilletes.zip%cd ProyectoBilletes

In [ ]:
!pip install -q pillow-heifimport syssys.path.insert(0, 'src')print('Python', sys.version.split()[0])

## 2. Verificar el banco de datos

In [ ]:
import os, globfor raiz in ['datos/autentico', 'datos/reproduccion']:    for sub in sorted(glob.glob(os.path.join(raiz, '*'))):        n = len(glob.glob(os.path.join(sub, '*')))        print(f'{sub:38s} {n:3d} imágenes')

## 3. Condiciones de adquisiciónSe documenta la resolución espacial de cada archivo, que es el parámetro quecondiciona la detectabilidad de la trama de semitono: el criterio de Nyquistexige al menos dos muestras por periodo de trama.

In [ ]:
from pdi import io_utils as IOimport numpy as npfrom PIL import Imagetry:    import pillow_heif; pillow_heif.register_heif_opener()except Exception:    passfilas = []for r in IO.listar_imagenes('datos/autentico'):    im = Image.open(r)    filas.append((os.path.basename(os.path.dirname(r)),                  os.path.splitext(r)[1].lower(), im.size[0], im.size[1]))import collectionsfor clase, ext, w, h in sorted(set(filas)):    passcnt = collections.Counter((c, e, w, h) for c, e, w, h in filas)for (c, e, w, h), n in sorted(cnt.items()):    print(f'{c:8s} {e:6s} {w}x{h} px   {n:3d} archivos   {w*h/1e6:.1f} Mpx')

## 4. Cadena de procesamiento sobre una imagenSegmentación por distancia de Mahalanobis sobre cromaticidad, umbralizaciónde Otsu con histéresis, morfología, componente conexa mayor, envolventeconvexa, cuadrilátero de área máxima y rectificación por homografía.

In [ ]:
import matplotlib.pyplot as pltfrom pdi import segmentacion as SEGruta = sorted(IO.listar_imagenes('datos/autentico/50mil'))[0]img = IO.reducir_lado_mayor(IO.cargar_rgb(ruta), 1024)et = SEG.procesar(img)print('Umbral de Otsu            :', round(et['umbral'], 4))print('Error de reproyección (px):', round(et['error_reproyeccion'], 5))print('Rectangularidad           :', round(et['diagnostico']['rectangularidad'], 4))print('\nMatriz de homografía H:')print(np.round(et['homografia'], 6))

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(14, 7.5)); ax = ax.ravel()ax[0].imshow(img.astype(np.uint8));            ax[0].set_title('(a) Original')ax[1].imshow(et['saliencia'], cmap='inferno'); ax[1].set_title('(b) Saliencia')ax[2].imshow(et['binaria'], cmap='gray');      ax[2].set_title('(c) Histéresis')ax[3].imshow(et['componente'], cmap='gray');   ax[3].set_title('(d) Componente mayor')ax[4].imshow(img.astype(np.uint8))q = np.vstack([et['esquinas'], et['esquinas'][:1]])ax[4].plot(q[:,0], q[:,1], '-o', color='#f33', lw=2); ax[4].set_title('(e) Cuadrilátero')ax[5].imshow(et['rectificada'].astype(np.uint8)); ax[5].set_title('(f) Rectificada')for a in ax: a.set_xticks([]); a.set_yticks([])plt.tight_layout(); plt.show()

## 5. Generar reproducciones simuladas**Advertencia metodológica.** Estas imágenes no son fotografías de billetesreproducidos físicamente: simulan el efecto de la cadena digitalización →impresión de consumo → recaptura. Sirven como prueba de concepto deldescriptor espectral y no sustituyen la validación sobre ejemplares reales.

In [ ]:
!python scripts/simular_reproduccion.py datos/autentico datos/simulado 6.0

## 6. Comparación espectral: auténtico contra copia

In [ ]:
from pdi import color as C, descriptores as D, clasificador as CLr_sim = ruta.replace('autentico', 'simulado')r_sim = os.path.splitext(r_sim)[0] + '_SIM.png'et2 = SEG.procesar(IO.reducir_lado_mayor(IO.cargar_rgb(r_sim), 1024))fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))for rect, etq, est in ((et['rectificada'], 'Auténtico', '-'),                       (et2['rectificada'], 'Reproducción simulada', '--')):    g = C.rgb_a_gris(rect) / 255.0    p = np.mean([D.perfil_radial(r) for r in CL.extraer_rois(g)], axis=0)    ax[0].semilogy(np.linspace(0, 1, len(p)), p / p.max(), est, lw=1.8, label=etq)    razon, pico = D.energia_radial(CL.extraer_rois(g)[0])    print(f'{etq:24s}  E_rho = {razon:.4f}   pico = {pico:7.1f}')ax[0].set_xlabel(r'Frecuencia radial normalizada $\rho$')ax[0].set_ylabel('Potencia relativa'); ax[0].legend(); ax[0].grid(alpha=.3)ax[0].set_title('Perfil radial de potencia')roi = CL.extraer_rois(C.rgb_a_gris(et2['rectificada'])/255.0)[0]ax[1].imshow(np.log1p(D.espectro_magnitud(roi)), cmap='magma')ax[1].set_title('Espectro de la copia (escala log)')ax[1].set_xticks([]); ax[1].set_yticks([])plt.tight_layout(); plt.show()

## 7. Entrenamiento y evaluación

In [ ]:
!python src/entrenar.py

## 8. Resultados

In [ ]:
import jsonr = json.load(open('resultados/resultados.json'))print(f"Segmentación válida : {r['segmentacion_ok']}/{r['n_imagenes']} "      f"({100*r['tasa_segmentacion']:.1f} %)")print(f"Denominación        : {100*r['exactitud_denominacion']:.1f} % "      f"(dejando uno fuera, n={r['n_den_loo']})\n")print('Matriz de confusión de denominación')print('filas = real, columnas = predicho ->', r['etiquetas_confusion'])print(np.array(r['matriz_confusion_denominacion']), '\n')if 'ablacion' in r:    print(f"{'Descriptores':<14}{'Exac.':>8}{'F1':>8}{'FNR':>8}{'AUC':>8}")    for k, v in r['ablacion'].items():        print(f"{k:<14}{100*v['exactitud']:7.1f}%{v['f1']:8.3f}"              f"{v['FNR']:8.3f}{v['auc']:8.3f}")

## 9. Barrido de dificultadCuanto más fina es la trama de la impresora, más se desplaza su energía haciala frecuencia de Nyquist del sensor y más difícil resulta detectarla. Esteexperimento acota el rango de validez del método.

In [ ]:
!python scripts/barrido_dificultad.py

## 10. Prototipo de la interfazEl servidor de la interfaz accesible está pensado para ejecutarse en uncomputador de la misma red Wi-Fi que el teléfono. En Colab no hay red localcompartida, de modo que esta celda solo verifica que el servidor levanta yque el modelo carga correctamente.Para probarlo en el Samsung Galaxy S24 FE, ejecútelo localmente:```python src/app.py --host 0.0.0.0 --puerto 5000```y abra `http://<IP-del-computador>:5000` en el teléfono.

In [ ]:
import subprocess, time, urllib.requestp = subprocess.Popen(['python', 'src/app.py', '--host', '127.0.0.1',                      '--puerto', '5056'],                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT)time.sleep(8)try:    codigo = urllib.request.urlopen('http://127.0.0.1:5056/', timeout=10).status    print('El servidor responde. Código HTTP:', codigo)finally:    p.terminate()